In [1]:
import pandas as pd
import numpy as np

from tqdm import trange

In [2]:
TOP_ANIME_PATH = "top_animes.csv"
ANIME_TRAIN_TEST_SPLIT = 0.8
MIN_SHOWS  = 50
SEEDS = 1400,1997

In [3]:
import kagglehub
svanoo_myanimelist_dataset_path = kagglehub.dataset_download('svanoo/myanimelist-dataset')
user = pd.read_csv(f'{svanoo_myanimelist_dataset_path}/user.csv',
                   sep = '\t', keep_default_na=False,usecols=("user_id","num_completed","mean_score")
                   ,index_col = "user_id")
top_animes = pd.read_csv(TOP_ANIME_PATH,keep_default_na=False,index_col="anime_id")
top_animes["number"] = np.arange(top_animes.shape[0])
user["number_reviews"] = np.zeros(user.shape[0],dtype=int)

In [4]:
NUM_USER_ANIME_FILE=70
cols_to_use = ['anime_id', 'user_id', 'score', 'favorite']
for x in trange(NUM_USER_ANIME_FILE):
    file_name = "{}/user_anime{:012d}.csv".format(svanoo_myanimelist_dataset_path,x)
    df = pd.read_csv(
        file_name,
        sep='\t',
        keep_default_na=False,
        na_values=("-------",""),
        low_memory=False,
        usecols=cols_to_use  # Only read needed columns
    ).dropna()
    df = df[df["anime_id"].isin(top_animes.index)]
    df= df[df["user_id"].isin(user.index)]
    df_users=df.value_counts("user_id")
    user.loc[df_users.index,'number_reviews'] += df_users

100%|██████████| 70/70 [04:18<00:00,  3.69s/it]


In [5]:
users = user[user["number_reviews"]>=MIN_SHOWS]
train_users = pd.read_csv("train_users.csv",index_col="user_id")
validation_users = pd.read_csv("validation_users.csv",index_col="user_id")
users=users.drop(train_users.index).drop(validation_users.index)
test_users = users.sample(n=validation_users.shape[0]//2,random_state = SEEDS[0])
test_users = pd.DataFrame(index = test_users.index,
                           data=np.arange(test_users.shape[0]))

In [6]:
print(test_users.head())
print(test_users.shape)

              0
user_id        
theobus       0
trananhduy    1
zskyar        2
mantaemi      3
nineteenling  4
(5794, 1)


In [7]:
def normalize(x,in_place=True):
    t = x!=0
    means = x.sum(axis=-1)/t.sum(axis=-1)
    new_x = np.where(t,x - means[:,np.newaxis],0)
    return new_x#/np.linalg.norm(x,axis=-1)[:,np.newaxis]

In [8]:
class MaximumStructure:
    # INITIAL = 200
    def __init__(self,keys):
        keep=keys>0
        self.inds = np.arange(keys.shape[0])
        self.inds= self.inds[keep]
        self.keys = keys[keep]
        self.shape = self.keys.shape[0]
        self.split = self.shape
        self.heapify()

    def swap(self,i,j):
        self.keys[i],self.keys[j] = self.keys[j],self.keys[i]
        self.inds[j],self.inds[i]=self.inds[i],self.inds[j]

    def sift_down(self,i):
        val_cur = self.keys[i]
        if 2*i+1<self.split:
            left_cur = self.keys[2*i+1]
        else:
            left_cur=-np.inf
        right_cur= self.keys[2*i+2] if 2*i+2<self.split else -np.inf
        if val_cur<right_cur and left_cur<=right_cur:
            selected=2*i+2
        elif val_cur<left_cur:
            selected=2*i+1
        else:
            return
        self.swap(i,selected)
        self.sift_down(selected)

    def heapify(self):
        # i = self.keys.argpartition(MaximumStructure.INITIAL)
        # self.keys=self.keys[i]
        # self.inds=self.inds[i]
        # i=np.argsort(self.keys[-MaximumStructure.INITIAL:])
        # self.keys[- MaximumStructure.INITIAL:]=self.keys[i]
        # self.inds[- MaximumStructure.INITIAL:]=self.inds[i]
        # self.split = self.shape - MaximumStructure.INITIAL
        for j in range((self.split-1)//2,-1,-1):
            self.sift_down(j)


    def extract_max(self):
        self.split-=1
        self.swap(0,self.split)
        self.sift_down(0)
        return self.inds[self.split],self.keys[self.split]

    def extracted(self):
        return self.inds[self.split:],self.keys[self.split:]

    def not_extracted(self):
        return self.split!=0

    def __iter__(self):
        for i in range(self.shape-1,self.split-1,-1):
            yield self.inds[i],self.keys[i]
        while self.split>-self.shape:
            yield self.extract_max()

In [9]:
test_values = np.zeros((test_users.shape[0],top_animes.shape[0]),dtype=np.float32)
NUM_USER_ANIME_FILE=70
cols_to_use = ['anime_id', 'user_id', 'score', 'favorite']
for x in trange(NUM_USER_ANIME_FILE):
    file_name = "{}/user_anime{:012d}.csv".format(svanoo_myanimelist_dataset_path,x)
    df = pd.read_csv(
        file_name,
        sep='\t',
        keep_default_na=False,
        na_values=("-------",""),
        low_memory=False,
        usecols=cols_to_use  # Only read needed columns
    ).dropna()
    df = df[df["anime_id"].isin(top_animes.index)]
    df= df[df["user_id"].isin(test_users.index)]
    us = df["user_id"].values
    user_numbers = test_users.loc[us,0].values
    anime_numbers = top_animes.loc[df["anime_id"].values,"number"]
    test_values[user_numbers,anime_numbers] = df["score"].values

100%|██████████| 70/70 [03:51<00:00,  3.31s/it]


In [10]:
normalized_test_values = normalize(test_values)

In [11]:
print(test_values[:10,:10])
print(normalized_test_values[:10,:10])

[[ 0.  0.  6.  0.  7.  0.  0.  7.  0.  9.]
 [ 0.  0.  0.  0.  0. 10.  0.  0.  0. 10.]
 [ 0. 10.  9. 10.  0.  0.  0.  8.  0.  7.]
 [10. 10.  9. 10. 10.  9.  9.  9.  9. 10.]
 [ 9.  8.  7.  9.  8.  8.  9.  0.  0. 10.]
 [ 7.  7.  3.  0.  0.  0.  0.  0.  0.  0.]
 [10.  0. 10. 10.  9.  9.  0.  0. 10. 10.]
 [ 9.  9.  0.  7. 10.  0.  9.  4.  0.  0.]
 [10. 10.  7. 10. 10.  8. 10.  8.  8.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0. 10.]]
[[ 0.          0.         -1.15151515  0.         -0.15151515  0.
   0.         -0.15151515  0.          1.84848485]
 [ 0.          0.          0.          0.          0.          0.796875
   0.          0.          0.          0.796875  ]
 [ 0.          1.3964497   0.3964497   1.3964497   0.          0.
   0.         -0.6035503   0.         -1.6035503 ]
 [ 2.07954545  2.07954545  1.07954545  2.07954545  2.07954545  1.07954545
   1.07954545  1.07954545  1.07954545  2.07954545]
 [ 0.45901639 -0.54098361 -1.54098361  0.45901639 -0.54098361 -0.54098361
   0.45901639

In [12]:
train_anime=top_animes.sample(frac=ANIME_TRAIN_TEST_SPLIT,random_state=SEEDS[1])
train_anime_bool = top_animes.index.isin(train_anime.index)
test_anime = top_animes[~train_anime_bool]

In [13]:
train_vals = np.load("training_vals.npy")
print(train_vals[0])

[ 0.1202057   0.07623785 -0.01169787 ...  0.          0.
  0.        ]


In [14]:
squared_error = 0
size=test_values.shape[0]
rating_inner_product = 0
rating_sq_norm = 0
samples_taken = 0
for i in trange(size):
    test_cur = normalized_test_values[i]
    test_cur_to_search = np.where(train_anime_bool,test_cur,0)

    t=test_cur_to_search!=0
    cur_train_vals = train_vals[:,t]
    similarities = np.inner(cur_train_vals,test_cur_to_search[t])#/np.linalg.norm(cur_train_vals,axis=1)
    similarities = MaximumStructure(similarities)
    ratings = np.zeros(top_animes.shape[0])
    for ind,row in test_anime.iterrows():
        anime_number = row["number"]
        act_rate = test_cur[anime_number]
        if act_rate==0:
            continue
        # t = act_rate!=0
        # anime = anime[t]
        # act_rate=act_rate[t]
        # ratings = train_vals[:,anime]
        # bool_anime  = ratings == 0
        # altered_similiarity = np.where(bool_anime,0,similarities[:,np.newaxis])
        # samples_taken+=anime.shape[0]
        # indices = np.argpartition(altered_similiarity,similarities.shape[0] -AMOUNT, axis=0)[-AMOUNT:]
        # sims = altered_similiarity[indices.T]
        # rates = ratings[indices.T]
        for i,k in similarities:
            rat = train_vals[i,anime_number]
            if rat!=0:
                break
        else:
            continue
        ratings[anime_number] = rat
    rating_inner_product+=np.inner(ratings,test_cur)
    rating_sq_norm += np.inner(ratings,ratings)

100%|██████████| 5794/5794 [56:40<00:00,  1.70it/s]  


In [15]:
test_users_test_anime = normalized_test_values[:size//5,~train_anime_bool]
norm = np.linalg.norm(test_users_test_anime,ord="fro")
cosine = rating_inner_product/(norm*np.sqrt(rating_sq_norm))

In [16]:
print(cosine)

0.7952118835815147


In [17]:
print(0)

0


In [18]:
size=test_values.shape[0]
rating_inner_product = 0
rating_sq_norm = 0
for i in trange(size):
    test_cur = test_values[i]
    t= test_cur !=0
    m = test_cur[t].sum()/t.sum()
    test_cur = normalized_test_values[i]
    for ind,row in test_anime.iterrows():
        anime_number = row["number"]
        act_rate = test_cur[anime_number]
        if act_rate==0:
            continue
        # t = act_rate!=0
        # anime = anime[t]
        # act_rate=act_rate[t]
        # ratings = train_vals[:,anime]
        # bool_anime  = ratings == 0
        # altered_similiarity = np.where(bool_anime,0,similarities[:,np.newaxis])
        # samples_taken+=anime.shape[0]
        # indices = np.argpartition(altered_similiarity,similarities.shape[0] -AMOUNT, axis=0)[-AMOUNT:]
        # sims = altered_similiarity[indices.T]
        # rates = ratings[indices.T]
        rat = row["score"] - m
        rating_inner_product+= rat*act_rate
        rating_sq_norm += rat*rat

100%|██████████| 5794/5794 [02:13<00:00, 43.29it/s]


In [19]:
print(rating_inner_product/(np.sqrt(rating_sq_norm)*
                            np.linalg.norm(normalized_test_values[:,~train_anime_bool])))

0.3120013996397356


In [20]:
train_values = np.load("training_vals.npy")
means = train_values.mean(axis=0)
del train_values

In [21]:
size=test_values.shape[0]
rating_inner_product = 0
rating_sq_norm = 0
for i in trange(size):
    test_cur = normalized_test_values[i]
    for ind,row in test_anime.iterrows():
        anime_number = row["number"]
        act_rate = test_cur[anime_number]
        if act_rate==0:
            continue
        # t = act_rate!=0
        # anime = anime[t]
        # act_rate=act_rate[t]
        # ratings = train_vals[:,anime]
        # bool_anime  = ratings == 0
        # altered_similiarity = np.where(bool_anime,0,similarities[:,np.newaxis])
        # samples_taken+=anime.shape[0]
        # indices = np.argpartition(altered_similiarity,similarities.shape[0] -AMOUNT, axis=0)[-AMOUNT:]
        # sims = altered_similiarity[indices.T]
        # rates = ratings[indices.T]
        rat = means[row["number"]]
        rating_inner_product+= rat*act_rate
        rating_sq_norm += rat*rat

100%|██████████| 5794/5794 [02:10<00:00, 44.31it/s]


In [22]:
print(rating_inner_product/(np.sqrt(rating_sq_norm)*
                            np.linalg.norm(normalized_test_values)))

0.15174203773660547
